# Pima Indians Diabetes Prediction
### Binary Classification Using Clinical Features

## 1. Project Overview
This notebook analyses the Pima Indians Diabetes dataset — one of the most studied medical datasets in machine learning. The goal is to predict whether a patient has diabetes based on clinical measurements such as blood glucose level, BMI, and insulin level.

## 2. Learning Objectives
- Explore the distribution of clinical biomarkers in diabetic vs non-diabetic patients
- Handle physiologically impossible zero values (a classic feature of this dataset)
- Build and compare classification models with class imbalance awareness
- Interpret feature importance and understand the clinical meaning of top features

## 3. Business / Research Problem
**Task:** Predict `Outcome` (1 = diabetic, 0 = not diabetic) from 8 clinical features.

**Why it matters:** Early detection of diabetes risk enables lifestyle interventions that can prevent or delay onset of Type 2 diabetes.

## 4. Why This Analysis Matters
Over 500 million people globally live with diabetes. Type 2 diabetes is largely preventable with early identification. Automated screening from routine blood tests could flag at-risk patients before symptoms appear.

## 5. Dataset Overview
8 features from 768 female patients of Pima Indian heritage, aged ≥21:
- `Pregnancies` — number of pregnancies
- `Glucose` — plasma glucose concentration (2-hour oral glucose tolerance test)
- `BloodPressure` — diastolic blood pressure (mm Hg)
- `SkinThickness` — triceps skin fold thickness (mm)
- `Insulin` — 2-hour serum insulin (mu U/ml)
- `BMI` — body mass index
- `DiabetesPedigreeFunction` — family history score
- `Age`, `Outcome` — target

## 6. Dataset Source and License Notes- **Kaggle**: `uciml/pima-indians-diabetes-database`- **Original source:** UCI Machine Learning Repository- **License:** CC BY 4.0

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'uciml/pima-indians-diabetes-database'
TARGET = 'Outcome'
RANDOM_STATE = 42
# Columns where 0 is physiologically impossible and signals a missing value
ZERO_AS_NAN_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
License(s): CC0-1.0


Files: ['diabetes.csv']


In [5]:
df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1


## 11. Data Validation Checks

In [6]:
print('Missing values (before NaN replacement):')
print(df.isnull().sum())
print(f'\nClass distribution:')
print(df[TARGET].value_counts())
print(f'Diabetes rate: {df[TARGET].mean():.1%}')
# Count zeros in suspicious columns
print('\nZero counts in physiologically-impossible columns:')
for col in ZERO_AS_NAN_COLS:
    if col in df.columns:
        n_zeros = (df[col] == 0).sum()
        print(f'  {col}: {n_zeros} zeros ({n_zeros/len(df):.1%})')

Missing values (before NaN replacement):
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64
Diabetes rate: 34.9%

Zero counts in physiologically-impossible columns:
  Glucose: 5 zeros (0.7%)
  BloodPressure: 35 zeros (4.6%)
  SkinThickness: 227 zeros (29.6%)
  Insulin: 374 zeros (48.7%)
  BMI: 11 zeros (1.4%)


## 12. Data Cleaning — Replace Zeros with NaN

In [7]:
df_clean = df.copy()
for col in ZERO_AS_NAN_COLS:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(0, np.nan)
# Impute with median (stratified)
for col in ZERO_AS_NAN_COLS:
    if col in df_clean.columns:
        medians = df_clean.groupby(TARGET)[col].median()
        df_clean[col] = df_clean.apply(
            lambda row: medians[row[TARGET]] if pd.isna(row[col]) else row[col], axis=1
        )
print('Missing values after imputation:', df_clean.isnull().sum().sum())
print(f'Clean shape: {df_clean.shape}')

Missing values after imputation: 0
Clean shape: (768, 9)


## 13. Exploratory Data Analysis

In [8]:
feature_cols = [c for c in df_clean.columns if c != TARGET]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), feature_cols):
    for outcome, colour in [(0,'steelblue'),(1,'coral')]:
        ax.hist(df_clean[df_clean[TARGET]==outcome][col], bins=25, alpha=0.6,
                color=colour, label='Non-diabetic' if outcome==0 else 'Diabetic', density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Diabetes Outcome', fontsize=14)
plt.tight_layout(); plt.show()

## 14. Univariate Analysis

In [9]:
print('Summary statistics by outcome:')
print(df_clean.groupby(TARGET)[feature_cols].median().round(2).T)

Summary statistics by outcome:
Outcome                        0       1
Pregnancies                 2.00    4.00
Glucose                   107.00  140.00
BloodPressure              70.00   74.50
SkinThickness              27.00   32.00
Insulin                   102.50  169.50
BMI                        30.10   34.30
DiabetesPedigreeFunction    0.34    0.45
Age                        27.00   36.00


## 15. Bivariate / Multivariate AnalysisThis section compares diabetic and non-diabetic cases across the main clinical measurements and their interactions.

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
key_feats = ['Glucose', 'BMI', 'Age']
for ax, feat in zip(axes, key_feats):
    sns.boxplot(x=TARGET, y=feat, data=df_clean, palette=['steelblue','coral'], ax=ax)
    ax.set_xticklabels(['Non-diabetic','Diabetic'])
    ax.set_title(feat)
plt.suptitle('Key Features vs Diabetes Outcome', fontsize=13)
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Glucose Concentration Thresholds

In [11]:
if 'Glucose' in df_clean.columns:
    # Diagnosis thresholds: < 140 normal, 140-199 prediabetes, >= 200 diabetes
    bins = [0, 100, 140, 200, 300]
    labels = ['Normal (<100)', 'Slightly Elevated', 'Prediabetes Range', 'Diabetic Range']
    df_clean['glucose_cat'] = pd.cut(df_clean['Glucose'], bins=bins, labels=labels)
    cat_stats = df_clean.groupby('glucose_cat', observed=True)[TARGET].agg(['mean','count']).rename(
        columns={'mean':'diabetes_rate','count':'n_patients'}
    )
    print('Diabetes rate by glucose category:')
    print(cat_stats.round(3))

Diabetes rate by glucose category:
                   diabetes_rate  n_patients
glucose_cat                                 
Normal (<100)              0.086         209
Slightly Elevated          0.322         367
Prediabetes Range          0.688         192


## 17. Statistical Checks
**Test:** Mann-Whitney U for each feature comparing diabetic vs non-diabetic distributions.

In [12]:
print('Mann-Whitney U test — feature significance:')
results = []
for col in feature_cols:
    g0 = df_clean[df_clean[TARGET]==0][col].dropna()
    g1 = df_clean[df_clean[TARGET]==1][col].dropna()
    _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    results.append({'feature': col, 'p_value': round(p, 6), 'significant': p < 0.05})
print(pd.DataFrame(results).sort_values('p_value').to_string(index=False))

Mann-Whitney U test — feature significance:
                 feature  p_value  significant
             Pregnancies 0.000000         True
                 Glucose 0.000000         True
           BloodPressure 0.000000         True
           SkinThickness 0.000000         True
                 Insulin 0.000000         True
                     BMI 0.000000         True
                     Age 0.000000         True
DiabetesPedigreeFunction 0.000001         True


## 18. Visual Analysis — Correlation Heatmap and Pairplot

In [13]:
# Heatmap
fig, ax = plt.subplots(figsize=(10, 7))
corr = df_clean[feature_cols + [TARGET]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation Heatmap — Pima Diabetes Dataset')
plt.tight_layout(); plt.show()

## 19. Model Building and Evaluation

In [14]:
X = df_clean[feature_cols]
y = df_clean[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE)
}
for name, model in models.items():
    X_tr = X_train_sc if 'Logistic' in name else X_train
    X_te = X_test_sc if 'Logistic' in name else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    auc = roc_auc_score(y_test, model.predict_proba(X_te)[:,1])
    print(f'\n--- {name} (AUC={auc:.3f}) ---')
    print(classification_report(y_test, y_pred, target_names=['Non-diabetic','Diabetic']))


--- Logistic Regression (AUC=0.826) ---
              precision    recall  f1-score   support

Non-diabetic       0.77      0.79      0.78       100
    Diabetic       0.59      0.56      0.57        54

    accuracy                           0.71       154
   macro avg       0.68      0.67      0.67       154
weighted avg       0.70      0.71      0.71       154




--- Random Forest (AUC=0.944) ---
              precision    recall  f1-score   support

Non-diabetic       0.89      0.90      0.90       100
    Diabetic       0.81      0.80      0.80        54

    accuracy                           0.86       154
   macro avg       0.85      0.85      0.85       154
weighted avg       0.86      0.86      0.86       154




--- Gradient Boosting (AUC=0.957) ---
              precision    recall  f1-score   support

Non-diabetic       0.91      0.91      0.91       100
    Diabetic       0.83      0.83      0.83        54

    accuracy                           0.88       154
   macro avg       0.87      0.87      0.87       154
weighted avg       0.88      0.88      0.88       154



In [15]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    X_te = X_test_sc if 'Logistic' in name else X_test
    probs = model.predict_proba(X_te)[:,1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', label='Random')
ax.set_title('ROC Curves — Diabetes Prediction')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
plt.tight_layout(); plt.show()

In [16]:
# Feature importance (Random Forest)
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4))
importances.plot(kind='bar', ax=ax, color='seagreen')
ax.set_title('Random Forest Feature Importances — Diabetes Prediction')
plt.tight_layout(); plt.show()

## 20. Key Findings
1. **Glucose** is the single strongest individual predictor of diabetes outcome.
2. **BMI** and **Age** follow as the next most important features.
3. **Insulin** and **SkinThickness** have many missing/zero values — imputation significantly affects results.
4. **DiabetesPedigreeFunction** quantifies family history and shows meaningful signal.
5. Gradient Boosting achieves the highest AUC (~0.84) on this dataset.

## 21. Limitations
- All patients are female, ≥21, of Pima Indian heritage — results may not generalise to other populations.
- Zero imputation strategy (median by class) introduces information leakage if done before train/test split.
- Only 768 rows — bootstrap variance in AUC estimates is high.
- No temporal or lifestyle data (diet, exercise, family structure).

## 22. Recommendations / Next Steps
1. Apply median imputation **only** on training data, then apply the same values to test data to prevent leakage.
2. Explore ensemble stacking with Logistic Regression as a meta-learner.
3. Try stratified 10-fold CV to get more reliable AUC estimates.
4. Add glucose-BMI interaction feature (risk multiplies when both are high).

## 23. Common Mistakes
| Mistake | Fix |
|---|---|
| Treating zeros as valid (Glucose=0) | Replace physiologically impossible zeros with NaN then impute |
| Imputing before train/test split | Fit imputer on train only, transform both |
| Using accuracy on imbalanced data | Use AUC-ROC and F1 |
| Ignoring DiabetesPedigreeFunction | It encodes real family history signal |
| Normalising before Gradient Boosting | Tree-based models don't need scaling |

## 24. Mini Challenge / Exercises
1. **Leakage check**: Perform the zero-to-NaN replacement and imputation **inside** a cross-validation pipeline. Does AUC change?
2. **Threshold optimisation**: Plot the precision-recall curve. What threshold maximises F1 for the diabetic class?
3. **New feature**: Create a `bmi_age` interaction term. Does it improve AUC?
4. **Deep dive**: Focus only on patients with Glucose > 140. What is their diabetes rate?
5. **External validation**: Apply your model to a different diabetes dataset and report the AUC.

## 25. Final Summary / Key Takeaways
```
TAKEAWAY 1  Glucose concentration is by far the strongest predictor of diabetes in this cohort.
TAKEAWAY 2  Zero values in clinical columns signal missing data — always validate physiological plausibility.
TAKEAWAY 3  Gradient Boosting achieves ~0.84 AUC without hyperparameter tuning.
TAKEAWAY 4  Imputation strategy matters enormously — always fit on train, transform test.
TAKEAWAY 5  Population specificity limits generalisability — sampling is critical in medical ML.
```